# Notebook 1 : Union de Datos y Limpieza

Este notebook tiene como propósito preparar los datos para su análisis. 

Se realizaron los siguientes puntos:

1. Carga de datasets originales.
2. Detección y tratamiento de valores nulos.
3. Detección y eliminación de duplicados.
4. Corrección de tipos de datos.
5. Creación de nuevas columnas
6. Creación de categorías
7. Exportación del dataset limpio



## 1. Carga de datasets originales

- Cargar datasets original de World Bank

- Se carga en el DataFrame df utilizando la función read_csv de la librería Pandas. 
- Esta función permite importar archivos CSV y representarlos en un DataFrame, facilitando su manipulación y análisis.

- Skiprows

- Se usó skiprows=4  para que el archivo comience después de las cuatro primeras filas

In [1]:
import pandas as pd

df_bank = pd.read_csv("../Datos_Fuente/API_ST.INT.ARVL_DS2_en_csv_v2_346201.csv", skiprows=4)

df_bank.head() # Visualizacion


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"International tourism, number of arrivals",ST.INT.ARVL,NaN,NaN,NaN,NaN,NaN,NaN,...,1.863000e+06,1.897000e+06,1.951000e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,"International tourism, number of arrivals",ST.INT.ARVL,NaN,NaN,NaN,NaN,NaN,NaN,...,3.825835e+07,4.118915e+07,3.982670e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,"International tourism, number of arrivals",ST.INT.ARVL,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Africa Western and Central,AFW,"International tourism, number of arrivals",ST.INT.ARVL,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Angola,AGO,"International tourism, number of arrivals",ST.INT.ARVL,NaN,NaN,NaN,NaN,NaN,NaN,...,2.610000e+05,2.180000e+05,2.180000e+05,NaN,NaN,NaN,NaN,NaN,NaN,NaN


- Dejar solo los años que necesitamos y seleccion de columnas

- El dastset Bank tiene los siguientes años de 1960 a 2025.
- El dataset Booking tiene los siguientes años  2015, 2016 y 2017.
- Por eso filtramos solo los años que vamos a usar 2015, 2016 y 2017.
- Se selecciona algunas columnas  para nuestro dataframe.

In [2]:
df_bank= df_bank[["Country Name", "Country Code", "2015", "2016", "2017"]]

df_bank.head() # Visualización

,Country Name,Country Code,2015,2016,2017
0,Aruba,ABW,1.832000e+06,1.758000e+06,1.863000e+06
1,Africa Eastern and Southern,AFE,3.531868e+07,3.764589e+07,3.825835e+07
2,Afghanistan,AFG,NaN,NaN,NaN
3,Africa Western and Central,AFW,1.331168e+07,1.315078e+07,NaN
4,Angola,AGO,5.920000e+05,3.970000e+05,2.610000e+05


- Convertir columnas en filas

- "melt" se usa para transformar columnas en filas, de esta manera los años estarán en una sola columna.

- id_vars=["Country Name", "Country Code"]--> Mantener estas columnas no se hace nada
- value_vars=["2015", "2016", "2017"]--> Pasar de columnas a filas.
- var_name="year" --> Nombre de la nueva columna donde van los años
- value_name="arrivals" --> Se crea otra columna donde van los valores que estaban en cada uno de los años



In [3]:

df_bank_pre = df_bank.melt(
    id_vars=["Country Name", "Country Code"],
    value_vars=["2015", "2016", "2017"],
    var_name="year",
    value_name="arrivals"
)



- Conversión el tipo de columna

- La columna "year" la convierto a tipo entero usando "astype(int)" porque, después de aplicar "melt" los años quedan como texto (string). Si no se  convierten  a números, no se podrán usar..

- Despues la columna "arrivals" la convierto a valores numéricos con "pd.to_numeric()",  porque algunos valores pueden venir como texto o pueden tener algun símbolo. 

- Finalmente ponemos "errors="coerce" esto se pone cuando no se pueda convertir un valor que ponga "NaN".



In [4]:
df_bank_pre["year"] = df_bank_pre["year"].astype(int)
df_bank_pre["arrivals"] = pd.to_numeric(df_bank_pre["arrivals"], errors="coerce")

- Verificación

- Es importante saber una vez creada la columna " arrivals" saber cuantos datos faltan, para ello se usa .isna().mean().
- En nuestro caso  ha salido 10,18% que es un porcentaje aceptable (se explica con más detalle en el analisis de nulos).
- Por tanto la union con booking será valida.


In [5]:
df_bank_pre["arrivals"].isna().mean() * 100

np.float64(10.18867924528302)

- Renombrar columnas 

- Antes de la union de dataset bank con el dataset de booking es necesario que el nombre de las columnas se llamen igual. En booking ( country y arrival_date_year) y en bank ( Country Code y year).

- Se usa " rename" para que las columnas bank tengan el mismo nombre que las columna de booking, asi se evitaran  problemas en la unión.


In [6]:
df_bank_pre = df_bank_pre.rename(columns={
    "Country Code": "country","year": "arrival_date_year"})


- Verificamos el cambio de nombre

In [7]:
df_bank_pre[["country", "arrival_date_year", "arrivals"]].sample(10)

,country,arrival_date_year,arrivals
230,TEC,2015,2.783254e+08
634,IDB,2017,NaN
120,KEN,2015,1.181000e+06
654,KNA,2017,1.194000e+06
787,VUT,2017,3.327000e+05
244,TUV,2015,2.400000e+03
440,NLD,2016,1.582800e+07
543,AUS,2017,8.815000e+06
229,TEA,2015,2.296441e+08
599,ERI,2017,NaN


- Carga de datasets original de Booking y la librería

In [8]:
import pandas as pd

booking = pd.read_csv("../Datos_Fuente/hotel_bookings.csv")

- Unir los dos dos datasets ( booking y df_bank_pre) 

- Se une el dataset de booking con el del Banco Mundial ( df_bank_pre) mediante el uso "merge" , se llamará " df_final"
"
- La unión se hace por las columnas  de "country y arrival_date_year"  que son las que tienen en común.

- Se usa "left_on y  right_on" para indiar las columnas que deben coincidir en cada uno de los dataset para poder unirlos.

- Finalmente usamos" how="left" porque se quiere mantener  todas las filas del dataset de principal ( booking)

In [9]:
df_final = booking.merge(
    df_bank_pre,
    left_on=["country", "arrival_date_year"],
    right_on=["country", "arrival_date_year"],
    how="left"
)


- Verificación del requisito del proyecto

- El dataframe final ("df_final") tiene una forma de (119390, 34), significa que contiene 119.390 filas y 34 columnas. 

- Por tanto, cumple el requisito del proyecto, de al menos 50.000 filas y 20 columnas. 


In [10]:
df_final.shape

(119390, 34)

## 2. Detección y tratamiento de valores nulos.

- Se necesita valorar la calidad del dataset final.
- Valorar si esn su conjunto es valido hay o no hay muchos datos faltantes.
- Si hay datos faltantes en algunas columnas ver que se hace
- Como hemos visto se usa para ello ".isna().mean() * 100" 


In [11]:
df_final.isna().mean() * 100

hotel                              0.000000
is_canceled                        0.000000
lead_time                          0.000000
arrival_date_year                  0.000000
arrival_date_month                 0.000000
arrival_date_week_number           0.000000
arrival_date_day_of_month          0.000000
stays_in_weekend_nights            0.000000
stays_in_week_nights               0.000000
adults                             0.000000
children                           0.003350
babies                             0.000000
meal                               0.000000
country                            0.408744
market_segment                     0.000000
distribution_channel               0.000000
is_repeated_guest                  0.000000
previous_cancellations             0.000000
previous_bookings_not_canceled     0.000000
reserved_room_type                 0.000000
assigned_room_type                 0.000000
booking_changes                    0.000000
deposit_type                    

- Para poder tomar una decisión, he usado en esta tabla de porcentajes para poder tomar las decisiones( explicada en el informe) que dice: 

      - 5% / 5%-15%  Conservar (Children, country,Country Name, arrivals )
      - 15-30%  Conservar ( agent)
      - > 50%   Eliminar ( company)
- Por tanto, la columna que sobresale por tener el 94,30% de sus datos faltantes es la que se va a eliminar.

- Las otras columnas donde se ha obtenido algún procentajes se encuentran dentro de los margenes aceptables.

- Eliminar la columna " company" 

- drop(): eliminar la columna
- columns = la columna que hay que eliminar
- inplace=True: que el cambio lo aplique directamnete en nuestro df_final 

In [12]:
df_final.drop(columns=['company'], inplace=True)

- Verificación

- De esta manera se verifica si la columna existe o no 
     True: Existe
     False: No existe 
- Se verifica que la columna ya no esta

In [13]:
'company' in df_final.columns

False

## 3. Detección y eliminación de duplicados.

- Es otro de los pasos necesarios para saber si nuestro dastaset tiene filas repetidas.
- Se aplica ".duplicated()" si devuelve:
       False: Fila única
       True: Fila duplicada

In [14]:
df_final.duplicated()

0         False
1         False
2         False
3         False
4         False
          ...  
119385    False
119386    False
119387    False
119388    False
119389    False
Length: 119390, dtype: bool

- Verificación

- Se concluye que nuestro daset no tiene duplicados

## 4. Corrección de tipos de datos

 

- Se utiliza  ".info()" para obtener un resumen técnico del dataset final. 
 
 - Esta función muestra el número de filas y columnas, los nombres de cada columna, sus tipos de datos y la cantidad de valores no nulos. 

 - Esta información es útil para evaluar la estructura del dataframe, identificar posibles problemas como valores faltantes o tipos de datos incorrectos, y confirmar que el dataset está listo para el análisis.


In [15]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 33 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

- Correcciones 

- Columna " reservation_status_date" : Convertir de str a Datetime
- Columna " arrival_date_moth": Convertir de str a Int
- Columna "children", "agent":    Convertir de float a Int
- Columna " country":    Renombrar para evitar confusion a country_ISO-3

#### Convertir de str a  Datetime 

- Se convierte esta columna de formato tetxo a  formato fecha yyyy-mm-dd usndo pd.to_datetime.

In [16]:
df_final['reservation_status_date'] = pd.to_datetime(df_final['reservation_status_date'])

df_final['reservation_status_date'].dtype # Verificación [us] mircosegundos

dtype('<M8[us]')

#### Convertir de str  a Int

- Se  crea una lista  "month_list"  para almacenar los nombres de los meses en inglés se convierte la columna del mes de llegada de texto a número utilizando un diccionario que asigna a cada mes su valor del 1 al 12.




In [17]:
month_list = {
    'January': 1,
    'February': 2,
    'March': 3,
    'April': 4,
    'May': 5,
    'June': 6,
    'July': 7,
    'August': 8,
    'September': 9,
    'October': 10,
    'November': 11,
    'December': 12
}

df_final['arrival_month_num'] = df_final['arrival_date_month'].map(month_list)

df_final['arrival_month_num'].dtype # Verificación


dtype('int64')

#### Convertir de float a int

- Se usa " fillna(0)" para que reemplace  los valores faltantes con cero porque los NaN dan problemas en los calculos y graficas. 

- Se usa "astype(int)" para convertir la columna en numero entero.

- Se interpreta que si no hay información es porque no hay niños.


In [18]:
df_final['children'] = df_final['children'].fillna(0).astype(int)

df_final['children'].dtype # Verificación

dtype('int64')

In [19]:

df_final['agent'] = df_final['agent'].fillna(0).astype(int)

df_final['agent'].dtype # Verificación

dtype('int64')

#### Renombrar columna 

- Se renombro la columna country por country _ISO-3.
- Se usa " rename" que cambia el nombre.

In [20]:
df_final = df_final.rename(columns={'country': 'country_ISO-3'})


## 5. Creación de nuevas columnas

#### Columna Nueva " Continent"

- Se usa "unique"  para ver los paises que aparecen en el datset y asi poder crearme la lista por medio de los codigos ISO-3

In [21]:
df_final['country_ISO-3'].unique()


<StringArray>
['PRT', 'GBR', 'USA', 'ESP', 'IRL', 'FRA',   nan, 'ROU', 'NOR', 'OMN',
 ...
 'ATA', 'GTM', 'ASM', 'MRT', 'NCL', 'KIR', 'SDN', 'ATF', 'SLE', 'LAO']
Length: 178, dtype: str


- Se identifican los países que hay y se elabora una lista con sus nombres correspondientes. Esta lista permite construir el diccionario "continent_list" de forma precisa, asegurando que cada país este asignado con su continente.




In [22]:
continent_list = {
    # Europa
    'PRT':'Europe',
    'GBR':'Europe',
    'ESP':'Europe',
    'IRL':'Europe',
    'FRA':'Europe',
    'ROU':'Europe',
    'NOR':'Europe',

    # América
    'USA':'America',
    'GTM':'America',

    # África
    'MRT':'Africa',
    'SDN':'Africa',
    'SLE':'Africa',

    # Asia
    'OMN':'Asia',
    'LAO':'Asia',

    # Oceanía
    'ASM':'Oceania',
    'NCL':'Oceania',
    'KIR':'Oceania',

    # Antártida / Territorios especiales
    'ATA':'Antarctica',
    'ATF':'Antarctica'
}


- Se utiliza la función "map()" para generar la columna "continent", que clasifica cada registro según su región geográfica.


In [23]:
df_final['continent'] = df_final['country_ISO-3'].map(continent_list)


- Se convierte la columna "continent" a "category" 

In [24]:
df_final['continent'] = df_final['continent'].astype('category')

df_final[['country_ISO-3', 'continent']].head() # Visualización

,country_ISO-3,continent
0,PRT,Europe
1,PRT,Europe
2,GBR,Europe
3,GBR,Europe
4,GBR,Europe


#### Columna Nueva " room_change"


- Primero, creamos la columnas boolena (true y false) pára saber si hubo o no cambio mediante != esto sabemos si son o no diferentes

- Posteriormente, estos valores se transformaron en etiquetas más legibles (“change” y “no_change”).

- Por ultimo la columna se convirtió al tipo "category" para optimizar su uso en el análisis descriptivo.




In [25]:
df_final['room_change'] = (df_final['reserved_room_type'] != df_final['assigned_room_type'])


In [26]:
df_final['room_change'] = df_final['room_change'].map({True: 'change', False: 'no_change'})
df_final['room_change'] = df_final['room_change'].astype('category')


In [27]:
df_final[['reserved_room_type', 'assigned_room_type', 'room_change']].head()

,reserved_room_type,assigned_room_type,room_change
0,C,C,no_change
1,C,C,no_change
2,A,C,change
3,A,A,no_change
4,A,A,no_change


#### Columna Nueva " total_stay"

- Simplemente sumamos ambas variables las noches de entre semana y las de fin de semana.
- Se crea la columna " total_stay"

In [28]:
df_final['total_stay'] = (
    df_final['stays_in_weekend_nights'] +
    df_final['stays_in_week_nights']
)

df_final[['stays_in_weekend_nights', 'stays_in_week_nights', 'total_stay']].head() 

,stays_in_weekend_nights,stays_in_week_nights,total_stay
0,0,0,0
1,0,0,0
2,0,1,1
3,0,1,1
4,0,2,2


In [29]:
df_final[['total_stay']].dtypes # Verificación que sea tipo númerico


total_stay    int64
dtype: object

#### Columna Nueva " total_people"

- Se aplica el mismo codigo usado antes que es sumar las tres varaibles para obtener el numero total de personas.
- Se crea la variable " total_people"

In [30]:
df_final['total_people'] = (df_final['adults'] +df_final['children'] +df_final['babies'])


df_final[['adults', 'children', 'babies', 'total_people']].head()

,adults,children,babies,total_people
0,2,0,0,2
1,2,0,0,2
2,1,0,0,1
3,1,0,0,1
4,2,0,0,2


In [31]:
df_final[['total_people']].dtypes # Verificación que sea tipo númerico


total_people    int64
dtype: object

#### Columna Nueva " familiy"

- Se comprueba si hay niños o bebes para ello se usa el "> 0" junto con  el operador " or (|)" para decir si hay niños o bebes.
- El resultado de aplicar esto es una columna booleana, que para que se más legible se ha hecho true= es familia pero entonces pones "si" y false= si no es familia  pones " no".
- Finalmente lo pasasmos a category

In [32]:
df_final['family'] = ((df_final['children'] > 0) | (df_final['babies'] > 0))

In [33]:
df_final['family'] = df_final['family'].map({True: 'si', False: 'no'})

In [34]:

df_final['family'] = df_final['family'].astype('category')

df_final[['adults', 'children', 'babies', 'family']].tail() 

,adults,children,babies,family
119385,2,0,0,no
119386,3,0,0,no
119387,2,0,0,no
119388,2,0,0,no
119389,2,0,0,no


## 6. Creación de categorías

- Antes de convertir las columnas de tipo texto a variables categóricas, se utilizó el método "unique()" con el objetivo de revisar los valores distintos presentes en cada una de ellas.
- Este paso es importante porque permite comprobar que la columna contiene realmente etiquetas repetidas y no valores numéricos o información que requiera otro tipo de tratamiento


In [35]:
df_final["hotel"].unique()


<StringArray>
['Resort Hotel', 'City Hotel']
Length: 2, dtype: str

In [36]:
df_final["meal"].unique()

<StringArray>
['BB', 'FB', 'HB', 'SC', 'Undefined']
Length: 5, dtype: str

In [37]:
df_final["country_ISO-3"].unique()

<StringArray>
['PRT', 'GBR', 'USA', 'ESP', 'IRL', 'FRA',   nan, 'ROU', 'NOR', 'OMN',
 ...
 'ATA', 'GTM', 'ASM', 'MRT', 'NCL', 'KIR', 'SDN', 'ATF', 'SLE', 'LAO']
Length: 178, dtype: str

In [38]:
df_final["market_segment"].unique()

<StringArray>
[       'Direct',     'Corporate',     'Online TA', 'Offline TA/TO',
 'Complementary',        'Groups',     'Undefined',      'Aviation']
Length: 8, dtype: str

In [39]:
df_final["distribution_channel"].unique()

<StringArray>
['Direct', 'Corporate', 'TA/TO', 'Undefined', 'GDS']
Length: 5, dtype: str

In [40]:
df_final["reserved_room_type"].unique()

<StringArray>
['C', 'A', 'D', 'E', 'G', 'F', 'H', 'L', 'P', 'B']
Length: 10, dtype: str

In [41]:
df_final["assigned_room_type"].unique()

<StringArray>
['C', 'A', 'D', 'E', 'G', 'F', 'I', 'B', 'H', 'P', 'L', 'K']
Length: 12, dtype: str

In [42]:
df_final["deposit_type"].unique()


<StringArray>
['No Deposit', 'Refundable', 'Non Refund']
Length: 3, dtype: str

In [43]:
df_final["customer_type"].unique()

<StringArray>
['Transient', 'Contract', 'Transient-Party', 'Group']
Length: 4, dtype: str

In [44]:
df_final["reservation_status"].unique()

<StringArray>
['Check-Out', 'Canceled', 'No-Show']
Length: 3, dtype: str

In [45]:
df_final["Country Name"].unique()

<StringArray>
[      'Portugal', 'United Kingdom',  'United States',          'Spain',
        'Ireland',         'France',              nan,        'Romania',
         'Norway',           'Oman',
 ...
         'Guyana',      'St. Lucia',      'Guatemala', 'American Samoa',
     'Mauritania',  'New Caledonia',       'Kiribati',          'Sudan',
   'Sierra Leone',        'Lao PDR']
Length: 167, dtype: str

- Se definió una lista " columns_to_category" con las columnas que queremos aplicar la categoria.
- Se usa el bucle "for" para recorrer cada uno de los nombres de columnas dentro de la lista.
- Se aplica astype("category") para covertirlo en category

In [46]:
columns_to_category = ['hotel',"meal","country_ISO-3","market_segment","distribution_channel","reserved_room_type","assigned_room_type","deposit_type","customer_type","reservation_status","Country Name"]

for col in columns_to_category:
    df_final[col] = df_final[col].astype('category')


In [47]:
df_final.dtypes

hotel                                   category
is_canceled                                int64
lead_time                                  int64
arrival_date_year                          int64
arrival_date_month                           str
arrival_date_week_number                   int64
arrival_date_day_of_month                  int64
stays_in_weekend_nights                    int64
stays_in_week_nights                       int64
adults                                     int64
children                                   int64
babies                                     int64
meal                                    category
country_ISO-3                           category
market_segment                          category
distribution_channel                    category
is_repeated_guest                          int64
previous_cancellations                     int64
previous_bookings_not_canceled             int64
reserved_room_type                      category
assigned_room_type  

## 7. Exportación del dataset limpio

- Se exporta el dataset tratado utilizando "to_csv" que permite guarda el Dtaframe en formato csv.

In [48]:
df_final.to_csv('../Datos_Limpios/Proyecto_Final_Datos_Limpios.csv', index=False)

En esta fase me he sasegurado de que los datos estuvieran completos, consistentes y listos para el análisi descritivo y estadisticos.

In [49]:
df_final.to_excel('../Datos_Limpios/Proyecto_Final_Datos_Limpios.xlsx', index=False)